In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import os
from color_mapping import clean_color_column

file_path = os.path.expanduser("~/Desktop/regge.xlsx")
df = pd.read_excel(file_path)

model_cond = df[["Preis", "Zustand", "Seller Rating", "Farbe"]].copy()


def clean_price(x):
    if pd.isna(x):
        return np.nan
    x = (
        str(x)
        .replace("EUR", "")
        .replace("€", "")
        .replace(".", "")
        .replace(",", ".")
        .strip()
    )
    try:
        return float(x)
    except:
        return np.nan


def clean_rating(x):
    if pd.isna(x) or x == "N/A":
        return np.nan
    x = str(x).replace("%", "").replace(",", ".").strip()
    try:
        return float(x)
    except:
        return np.nan


model_cond["price"] = model_cond["Preis"].apply(clean_price)
model_cond["seller_rating"] = model_cond["Seller Rating"].apply(clean_rating)

model_cond["Zustand"] = model_cond["Zustand"].astype(str).str.lower().str.strip()
model_cond["Farbe"] = model_cond["Farbe"].astype(str).str.lower().str.strip()

model_cond["Zustand"] = model_cond["Zustand"].replace(
    {"neu": "new", "gebraucht": "used", "refurbished": "refurbished"}
)


model_cond["Farbe"] = clean_color_column(model_cond["Farbe"])

model_cond = model_cond.drop(columns=["Preis", "Seller Rating"])
model_cond = model_cond.dropna()

model_cond["Zustand"] = pd.Categorical(
    model_cond["Zustand"], categories=["new", "used", "refurbished"]
)

model_encoded = pd.get_dummies(
    model_cond, columns=["Zustand", "Farbe"], drop_first=True
)

X = model_encoded.drop("price", axis=1)
y = model_encoded["price"]

X = X.astype(float)
y = y.astype(float)

X = sm.add_constant(X)

model_condition = sm.OLS(y, X).fit()

print(model_condition.summary())

print("Observations:", len(y))
print("R²:", model_condition.rsquared)
print("Adjusted R²:", model_condition.rsquared_adj)

                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.232
Model:                            OLS   Adj. R-squared:                  0.204
Method:                 Least Squares   F-statistic:                     8.124
Date:                Fri, 08 May 2026   Prob (F-statistic):           1.25e-08
Time:                        13:28:47   Log-Likelihood:                -1127.9
No. Observations:                 196   AIC:                             2272.
Df Residuals:                     188   BIC:                             2298.
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                 791.4574    